In [1]:
from google.colab import files
from pathlib import Path
import json, math
import pandas as pd

uploaded = files.upload()
required = ["simulation_summary.json", "copper_simulation_results.csv"]
missing = [name for name in required if not Path(name).is_file()]
assert not missing, f"Missing required files: {missing}. Please upload them."

with open("simulation_summary.json", "r", encoding="utf-8") as file:
    summary = json.load(file)
data = pd.read_csv("copper_simulation_results.csv")
required_keys = [
    "equilibrium_volume_A3", "equilibrium_lattice_constant_A",
    "minimum_energy_eV", "bulk_modulus_GPa"
]
assert all(key in summary for key in required_keys), "Summary JSON has missing keys. Rerun Milestone 01 Step 3."
assert {"lattice_constant_A", "volume_A3", "energy_eV"}.issubset(data.columns), "Unexpected CSV columns."
assert len(data) >= 5, "CSV seems incomplete."

print("✅ Files loaded. You have", len(data), "simulation rows.")
print("ASE model:", summary.get("simulation_method", "ASE EMT"))
print("Equilibrium volume:", summary["equilibrium_volume_A3"], "Å³/atom")
display(data.head())

Saving 02_ai_scientific_benchmark_colab.ipynb to 02_ai_scientific_benchmark_colab (1).ipynb
✅ Files loaded. You have 15 simulation rows.
ASE model: ASE EMT
Equilibrium volume: 11.563251407120669 Å³/atom


,lattice_constant_A,volume_A3,energy_eV
0,3.300000,8.984250,0.357056
1,3.342857,9.338851,0.246559
2,3.385714,9.702663,0.159840
3,3.428571,10.075802,0.093977
4,3.471429,10.458387,0.045677


In [2]:

# STEP 2: Create Scientific Benchmark Questions

# Reference values from your uploaded JSON
v0 = float(summary["equilibrium_volume_A3"])
a0 = float(summary["equilibrium_lattice_constant_A"])
e0 = float(summary["minimum_energy_eV"])
b_gpa = float(summary["bulk_modulus_GPa"])

initial_volume = 3.6**3 / 4
volume_decrease = (
    (initial_volume - v0) / initial_volume * 100
)

numeric_questions = [
    {
        "id": "NUM-01",
        "question": "Equilibrium volume (A3/atom)?",
        "answer": v0,
        "tolerance": 0.05
    },
    {
        "id": "NUM-02",
        "question": "Equilibrium lattice constant (A)?",
        "answer": a0,
        "tolerance": 0.02
    },
    {
        "id": "NUM-03",
        "question": "Bulk modulus (GPa)?",
        "answer": b_gpa,
        "tolerance": 2.0
    },
    {
        "id": "NUM-04",
        "question": "Minimum energy (eV/atom)?",
        "answer": e0,
        "tolerance": 0.001
    },
    {
        "id": "NUM-05",
        "question": "Initial atomic volume (A3/atom)?",
        "answer": initial_volume,
        "tolerance": 0.05
    },
    {
        "id": "NUM-06",
        "question": "Percentage decrease in volume?",
        "answer": volume_decrease,
        "tolerance": 0.10
    }
]

print("Successfully created 6 scientific questions!")

for q in numeric_questions:
    print(q["id"], q["question"],
          "Reference:", round(q["answer"], 5))


Successfully created 6 scientific questions!
NUM-01 Equilibrium volume (A3/atom)? Reference: 11.56325
NUM-02 Equilibrium lattice constant (A)? Reference: 3.5896
NUM-03 Bulk modulus (GPa)? Reference: 132.6009
NUM-04 Minimum energy (eV/atom)? Reference: -0.0066
NUM-05 Initial atomic volume (A3/atom)? Reference: 11.664
NUM-06 Percentage decrease in volume? Reference: 0.86376


In [3]:

# STEP 3: Evaluate example AI answers

import pandas as pd
import math

def evaluate_ai_answer(ai_answer, correct_answer, tolerance):
    try:
        ai_answer = float(ai_answer)

        if not math.isfinite(ai_answer):
            return "FAIL"

        difference = abs(ai_answer - correct_answer)

        return "PASS" if difference <= tolerance else "FAIL"

    except (ValueError, TypeError):
        return "FAIL"


# Generate five correct example answers
demo_answers = {
    q["id"]: round(q["answer"], 5)
    for q in numeric_questions
}

# Introduce one intentional scientific error
demo_answers["NUM-03"] = 50.0

results = []

for q in numeric_questions:
    ai_answer = demo_answers[q["id"]]

    status = evaluate_ai_answer(
        ai_answer,
        q["answer"],
        q["tolerance"]
    )

    results.append({
        "Question": q["id"],
        "AI Answer": ai_answer,
        "Reference": round(q["answer"], 5),
        "Status": status
    })

result_df = pd.DataFrame(results)
display(result_df)

score = (result_df["Status"] == "PASS").sum()

print("=== AI EVALUATION REPORT ===")
print("Passed:", score)
print("Failed:", len(result_df) - score)
print("Accuracy:", round(score / len(result_df) * 100, 2), "%")


,Question,AI Answer,Reference,Status
0,NUM-01,11.56325,11.56325,PASS
1,NUM-02,3.58960,3.58960,PASS
2,NUM-03,50.00000,132.60090,FAIL
3,NUM-04,-0.00660,-0.00660,PASS
4,NUM-05,11.66400,11.66400,PASS
5,NUM-06,0.86376,0.86376,PASS


=== AI EVALUATION REPORT ===
Passed: 5
Failed: 1
Accuracy: 83.33 %


In [4]:

result_df.to_csv(
    "ai_numerical_evaluation_results.csv",
    index=False
)

from google.colab import files
files.download("ai_numerical_evaluation_results.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:

# STEP 4A: Scientific Reasoning Questions

import pandas as pd

v = float(summary["equilibrium_volume_A3"])
b = float(summary["bulk_modulus_GPa"])
a = float(summary["equilibrium_lattice_constant_A"])

reasoning_questions = [
    {
        "id": "REASON-01",
        "question": (
            f"The ASE EMT energy-volume curve for "
            f"FCC copper has a minimum near {v:.4f} "
            "A3/atom. Explain the U-shaped curve."
        ),
        "ai_answer": (
            "Compression below equilibrium increases "
            "repulsive interactions between atoms. "
            "Expansion weakens stabilizing interactions. "
            "The energy minimum represents an "
            "energetically favorable structure within EMT."
        )
    },
    {
        "id": "REASON-02",
        "question": (
            f"The fitted bulk modulus is {b:.2f} GPa. "
            "What does it mean? Is it experimental?"
        ),
        "ai_answer": (
            "Bulk modulus tells us how difficult "
            "it is to compress copper."
        )
    },
    {
        "id": "REASON-03",
        "question": (
            f"The equilibrium volume is {v:.4f} "
            "A3/atom. Is this the volume of the "
            "conventional FCC unit cell? Explain."
        ),
        "ai_answer": (
            "Yes. The reported volume is the "
            "complete conventional FCC unit cell. "
            "Its lattice constant is approximately "
            "2.26 angstroms."
        )
    }
]

reasoning_df = pd.DataFrame(reasoning_questions)

display(reasoning_df)

print("3 scientific reasoning questions created!")


,id,question,ai_answer
0,REASON-01,The ASE EMT energy-volume curve for FCC copper...,Compression below equilibrium increases repuls...
1,REASON-02,The fitted bulk modulus is 132.60 GPa. What do...,Bulk modulus tells us how difficult it is to c...
2,REASON-03,The equilibrium volume is 11.5633 A3/atom. Is ...,Yes. The reported volume is the complete conve...


3 scientific reasoning questions created!


In [6]:

# STEP 4B: Scientific Reasoning Evaluation

import pandas as pd

human_reviews = [
    {
        "id": "REASON-01",
        "correctness": 2,
        "evidence": 2,
        "clarity": 1,
        "feedback": "Correct scientific explanation."
    },
    {
        "id": "REASON-02",
        "correctness": 1,
        "evidence": 0,
        "clarity": 1,
        "feedback": "Incomplete explanation. EMT model context missing."
    },
    {
        "id": "REASON-03",
        "correctness": 0,
        "evidence": 0,
        "clarity": 1,
        "feedback": "Incorrect interpretation of FCC unit cell."
    }
]

review_df = pd.DataFrame(human_reviews)

review_df["total_score"] = (
    review_df["correctness"]
    + review_df["evidence"]
    + review_df["clarity"]
)

review_df["percentage"] = (
    review_df["total_score"] / 5 * 100
)

display(review_df)

print("=== SCIENTIFIC REASONING REPORT ===")
print("Questions evaluated:", len(review_df))
print("Average score:",
      round(review_df["total_score"].mean(), 2), "/ 5")


,id,correctness,evidence,clarity,feedback,total_score,percentage
0,REASON-01,2,2,1,Correct scientific explanation.,5,100.0
1,REASON-02,1,0,1,Incomplete explanation. EMT model context miss...,2,40.0
2,REASON-03,0,0,1,Incorrect interpretation of FCC unit cell.,1,20.0


=== SCIENTIFIC REASONING REPORT ===
Questions evaluated: 3
Average score: 2.67 / 5


In [7]:

# STEP 4C: Save Scientific Reasoning Evaluation

from google.colab import files

# Save evaluation results as CSV
review_df.to_csv(
    "ai_scientific_reasoning_evaluation.csv",
    index=False
)

print("Evaluation report saved successfully!")

# Download the CSV to your computer
files.download("ai_scientific_reasoning_evaluation.csv")


Evaluation report saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:

# STEP 5B: Evaluate a Real AI Model Response

import pandas as pd

# Verified ASE reference values
atomic_volume = float(
    summary["equilibrium_volume_A3"]
)

reference_cell_volume = 4 * atomic_volume
reference_lattice = reference_cell_volume ** (1/3)

# Values reported by Gemini
gemini_cell_volume = 46.253
gemini_lattice = 3.5896

# Numerical validation
cell_correct = abs(
    gemini_cell_volume - reference_cell_volume
) <= 0.01

lattice_correct = abs(
    gemini_lattice - reference_lattice
) <= 0.01

print("=== GEMINI SCIENTIFIC EVALUATION ===")
print("Reference cell volume:",
      round(reference_cell_volume, 4))
print("Gemini cell volume:", gemini_cell_volume)
print("Volume correct:", cell_correct)

print("Reference lattice constant:",
      round(reference_lattice, 4))
print("Gemini lattice constant:", gemini_lattice)
print("Lattice correct:", lattice_correct)

# Human review based on the submitted response
evaluation = {
    "Question_ID": "REASON-03",
    "AI_Model": "Google Gemini",
    "Scientific_Correctness": 2,
    "Scientific_Evidence": 2,
    "Clarity": 1,
    "Volume_Check": cell_correct,
    "Lattice_Check": lattice_correct,
    "Feedback": (
        "Correct FCC geometry, unit-cell volume, "
        "lattice constant, and scientific explanation."
    )
}

real_eval_df = pd.DataFrame([evaluation])

real_eval_df["Total_Score"] = (
    real_eval_df["Scientific_Correctness"]
    + real_eval_df["Scientific_Evidence"]
    + real_eval_df["Clarity"]
)

display(real_eval_df)

print("Final score:",
      real_eval_df["Total_Score"].iloc[0],
      "/ 5")


=== GEMINI SCIENTIFIC EVALUATION ===
Reference cell volume: 46.253
Gemini cell volume: 46.253
Volume correct: True
Reference lattice constant: 3.5896
Gemini lattice constant: 3.5896
Lattice correct: True


,Question_ID,AI_Model,Scientific_Correctness,Scientific_Evidence,Clarity,Volume_Check,Lattice_Check,Feedback,Total_Score
0,REASON-03,Google Gemini,2,2,1,True,True,"Correct FCC geometry, unit-cell volume, lattic...",5


Final score: 5 / 5


In [9]:

# FINAL STEP: Save Real AI Evaluation Report

from google.colab import files

# Export the evaluated Gemini response
real_eval_df.to_csv(
    "real_ai_model_evaluation.csv",
    index=False
)

print("Final AI evaluation report created!")
print("Model: Google Gemini")
print("Question: REASON-03")
print("Score: 5/5")
print("Numerical checks: PASSED")

# Download the report
files.download("real_ai_model_evaluation.csv")


Final AI evaluation report created!
Model: Google Gemini
Question: REASON-03
Score: 5/5
Numerical checks: PASSED


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>